# Thunderbird 350X Spare-Part Availability: Model Comparison

This notebook prepares the parts dataset, compares Logistic Regression, Decision Tree, and Random Forest, evaluates each model on the same stratified holdout set, and saves the best fitted pipeline.

**Important:** the dataset contains only 500 records and may be synthetic. Holdout scores are estimates, not proof of real-world accuracy. Validate with representative external data before using predictions in practice. Some input fields, such as estimated wait time and price, may encode information related to availability; interpret results accordingly.

## 1. Imports and project paths

The path setup works when the notebook is opened from the project root or run with the working directory set to the `ML` folder.

In [ ]:
from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "Dataset").is_dir() else CURRENT_DIR.parent
DATASET_PATH = PROJECT_ROOT / "Dataset" / "royal_enfield_thunderbird_350x_parts_dataset.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "availability_classifier.joblib"
SUMMARY_PATH = MODEL_DIR / "model_comparison_summary.json"

TARGET_COLUMN = "Availability status"
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_PATH}")

## 2. Load and inspect the dataset

Check the schema and data quality before training. Rows without a target label cannot be used for supervised learning, so they are removed explicitly.

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset file not found: {DATASET_PATH}")

df = pd.read_csv(DATASET_PATH)
required_columns = {
    "Manufacturing year",
    "Estimated wait days",
    "Estimated price (INR)",
    "Part name",
    "Part category",
    "Supplier type",
    "City",
    "Garage type",
    "Part condition",
    "Service priority",
    "Record date",
    TARGET_COLUMN,
}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

print(f"Dataset shape: {df.shape}")
display(df.head())
display(pd.DataFrame({
    "missing_values": df.isna().sum(),
    "data_type": df.dtypes.astype(str),
}))
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Rows with missing target: {df[TARGET_COLUMN].isna().sum()}")

df = df.dropna(subset=[TARGET_COLUMN]).copy()
print("\nAvailability class counts:")
display(df[TARGET_COLUMN].value_counts().rename_axis(TARGET_COLUMN).to_frame("count"))

## 3. Feature preparation

The record date is converted into month and year features. The original date column and target are excluded from model inputs. Numeric and categorical missing values are imputed inside the training pipeline to avoid learning preprocessing statistics from the test set.

In [ ]:
record_dates = pd.to_datetime(df["Record date"], errors="coerce")
df["record_month"] = record_dates.dt.month
df["record_year"] = record_dates.dt.year

FEATURE_COLUMNS = [
    "Manufacturing year",
    "Estimated wait days",
    "Estimated price (INR)",
    "Part name",
    "Part category",
    "Supplier type",
    "City",
    "Garage type",
    "Part condition",
    "Service priority",
    "record_month",
    "record_year",
]
NUMERIC_FEATURES = [
    "Manufacturing year",
    "Estimated wait days",
    "Estimated price (INR)",
    "record_month",
    "record_year",
]
CATEGORICAL_FEATURES = [
    column for column in FEATURE_COLUMNS if column not in NUMERIC_FEATURES
]

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")
print("Training class proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename("proportion").to_frame())
print("Test class proportions:")
display(y_test.value_counts(normalize=True).sort_index().rename("proportion").to_frame())

## 4. Build preprocessing and model pipelines

Every candidate uses identical preprocessing and the same train/test split. Unseen test categories are safely ignored by the one-hot encoder.

In [ ]:
def build_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ])


def build_pipeline(estimator):
    return Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", estimator),
    ])


candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample",
    ),
}
print("Candidates:", ", ".join(candidate_models))

## 5. Train and evaluate all candidates

Precision, recall, and F1 are weighted averages so they account for class support. The per-class classification report and confusion matrix are also retained for interpretation.

In [ ]:
class_labels = sorted(y.unique().tolist())
fitted_models = {}
comparison_rows = []
detailed_reports = {}
confusion_matrices = {}

for model_name, estimator in candidate_models.items():
    pipeline = build_pipeline(estimator)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    fitted_models[model_name] = pipeline
    comparison_rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision_weighted": precision_score(
            y_test, predictions, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(
            y_test, predictions, average="weighted", zero_division=0
        ),
        "f1_weighted": f1_score(
            y_test, predictions, average="weighted", zero_division=0
        ),
    })
    detailed_reports[model_name] = classification_report(
        y_test,
        predictions,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    confusion_matrices[model_name] = confusion_matrix(
        y_test, predictions, labels=class_labels
    )

comparison = pd.DataFrame(comparison_rows).sort_values(
    ["f1_weighted", "accuracy"], ascending=False
).reset_index(drop=True)
display(comparison.style.format({
    "accuracy": "{:.3f}",
    "precision_weighted": "{:.3f}",
    "recall_weighted": "{:.3f}",
    "f1_weighted": "{:.3f}",
}))

## 6. Confusion matrices and classification reports

Rows are actual labels and columns are predicted labels. Diagonal values are correct predictions; off-diagonal values show which classes are confused.

In [ ]:
fig, axes = plt.subplots(1, len(fitted_models), figsize=(18, 5))
for axis, (model_name, matrix) in zip(axes, confusion_matrices.items()):
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=class_labels,
        yticklabels=class_labels,
        ax=axis,
    )
    axis.set_title(model_name)
    axis.set_xlabel("Predicted label")
    axis.set_ylabel("Actual label")
plt.tight_layout()
plt.show()

for model_name, report in detailed_reports.items():
    print(f"\n{model_name} — per-class metrics")
    display(pd.DataFrame(report).T)

## 7. Select and save the winning pipeline

The winner is selected by weighted F1, using accuracy as the tie-breaker. Its pipeline is fitted on the training partition only; the holdout test set remains reserved for the reported comparison.

In [ ]:
best_row = comparison.iloc[0]
best_model_name = best_row["model"]
best_pipeline = build_pipeline(candidate_models[best_model_name])
best_pipeline.fit(X_train, y_train)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_pipeline, MODEL_PATH)

comparison_for_json = comparison.to_dict(orient="records")
summary = {
    "target_column": TARGET_COLUMN,
    "feature_columns": FEATURE_COLUMNS,
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "class_labels": [str(label) for label in class_labels],
    "selection_criterion": "weighted F1-score, then accuracy",
    "best_model": str(best_model_name),
    "best_model_metrics": {
        key: float(value)
        for key, value in best_row.items()
        if key != "model"
    },
    "comparison_metrics": comparison_for_json,
    "classification_reports": detailed_reports,
    "confusion_matrices": {
        name: matrix.tolist()
        for name, matrix in confusion_matrices.items()
    },
    "model_path": str(MODEL_PATH),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Selected model: {best_model_name}")
print(f"Weighted F1: {best_row['f1_weighted']:.3f}")
print(f"Accuracy: {best_row['accuracy']:.3f}")
print(f"Saved model pipeline: {MODEL_PATH}")
print(f"Saved comparison summary: {SUMMARY_PATH}")

## Interpretation

Use the comparison table together with the per-class reports and confusion matrices; accuracy alone can hide poor performance on less frequent availability classes. Because this dataset is small and may be synthetic, do not present these results as validated real-world performance. The saved model is the complete preprocessing-plus-classifier pipeline, so downstream code should pass input rows using the feature column names listed above.